In [1]:
from pathlib import Path
import os
import json
import math
import glob

import numpy as np
import pandas as pd

try:
    import geopandas as gpd
except ImportError:
    gpd = None

try:
    import rasterio
    from rasterio.transform import from_origin
except ImportError:
    rasterio = None

try:
    from scipy.interpolate import griddata
except ImportError:
    griddata = None

In [2]:
# --- main project folders ---
ROOT = Path("/home/jovyan")
RAW_DIR = ROOT / "raw_data"
DEPTH_GRID_DIR = RAW_DIR / "north_slope_depth_grids"

# where your parquet layers are / will be
PARQUET_DIRS = [
    ROOT / "parquets",
    ROOT / "processed_data",
    ROOT / "raw_data",
]

# output folder for rasters made from XYZ surfaces
RASTER_OUT_DIR = DEPTH_GRID_DIR / "rasters_from_xyz"
RASTER_OUT_DIR.mkdir(parents=True, exist_ok=True)

# manifest outputs
MANIFEST_DIR = ROOT / "project_manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

TXT_MANIFEST = MANIFEST_DIR / "north_slope_file_manifest.txt"
CSV_MANIFEST = MANIFEST_DIR / "north_slope_file_manifest.csv"

print("DEPTH_GRID_DIR:", DEPTH_GRID_DIR)
print("RASTER_OUT_DIR:", RASTER_OUT_DIR)
print("MANIFEST_DIR:", MANIFEST_DIR)


DEPTH_GRID_DIR: /home/jovyan/raw_data/north_slope_depth_grids
RASTER_OUT_DIR: /home/jovyan/raw_data/north_slope_depth_grids/rasters_from_xyz
MANIFEST_DIR: /home/jovyan/project_manifests


In [3]:
def scan_files(base_dirs, extensions=None):
    rows = []

    for base in base_dirs:
        if not base.exists():
            continue

        for path in base.rglob("*"):
            if path.is_file():
                if extensions:
                    if path.suffix.lower() not in extensions:
                        continue

                rows.append({
                    "name": path.name,
                    "suffix": path.suffix.lower(),
                    "parent": str(path.parent),
                    "full_path": str(path.resolve()),
                    "size_mb": round(path.stat().st_size / (1024**2), 3),
                    "modified": pd.to_datetime(path.stat().st_mtime, unit="s"),
                })

    df = pd.DataFrame(rows).sort_values(["suffix", "name", "modified"]).reset_index(drop=True)
    return df

base_dirs_to_scan = [
    ROOT,
]

manifest_df = scan_files(
    base_dirs_to_scan,
    extensions={".parquet", ".xyz", ".tif", ".tiff", ".ipynb", ".csv", ".txt", ".shp", ".geojson"}
)

manifest_df.head(20)
print(f"Found {len(manifest_df)} matching files.")

Found 804 matching files.


In [4]:
manifest_df.to_csv(CSV_MANIFEST, index=False)

with open(TXT_MANIFEST, "w", encoding="utf-8") as f:
    f.write("NORTH SLOPE PROJECT FILE MANIFEST\n")
    f.write("=" * 80 + "\n\n")

    for _, row in manifest_df.iterrows():
        f.write(f"NAME:     {row['name']}\n")
        f.write(f"TYPE:     {row['suffix']}\n")
        f.write(f"SIZE_MB:  {row['size_mb']}\n")
        f.write(f"MODIFIED: {row['modified']}\n")
        f.write(f"PARENT:   {row['parent']}\n")
        f.write(f"PATH:     {row['full_path']}\n")
        f.write("-" * 80 + "\n")

print("Saved:")
print(TXT_MANIFEST)
print(CSV_MANIFEST)

Saved:
/home/jovyan/project_manifests/north_slope_file_manifest.txt
/home/jovyan/project_manifests/north_slope_file_manifest.csv


In [5]:
parquet_df = manifest_df[manifest_df["suffix"] == ".parquet"].copy()
display(parquet_df[["name", "parent", "full_path", "size_mb"]].sort_values("name"))
print(f"Total parquet files: {len(parquet_df)}")

,name,parent,full_path,size_mb
330,alaska_base_map.parquet,/home/jovyan/notebooks/03_data_final/core_layers,/home/jovyan/notebooks/03_data_final/core_laye...,0.101
331,clean_2d_seismic.parquet,/home/jovyan/notebooks/03_data_final/core_layers,/home/jovyan/notebooks/03_data_final/core_laye...,4.930
332,clean_2d_seismic_features.parquet,/home/jovyan/notebooks/03_data_final/feature_l...,/home/jovyan/notebooks/03_data_final/feature_l...,2.884
333,clean_3d_seismic.parquet,/home/jovyan/notebooks/03_data_final/core_layers,/home/jovyan/notebooks/03_data_final/core_laye...,1.948
334,clean_3d_seismic_features.parquet,/home/jovyan/notebooks/03_data_final/feature_l...,/home/jovyan/notebooks/03_data_final/feature_l...,1.552
335,clean_well_locations.parquet,/home/jovyan/notebooks/03_data_final/core_layers,/home/jovyan/notebooks/03_data_final/core_laye...,0.192
336,clean_well_locations_features.parquet,/home/jovyan/notebooks/03_data_final/feature_l...,/home/jovyan/notebooks/03_data_final/feature_l...,0.966
337,clean_well_locations_features_dedup.parquet,/home/jovyan/notebooks/03_data_final/feature_l...,/home/jovyan/notebooks/03_data_final/feature_l...,0.819
338,north_slope_assessment_units.parquet,/home/jovyan/notebooks/03_data_final/core_layers,/home/jovyan/notebooks/03_data_final/core_laye...,1.034
339,north_slope_assessment_units_features.parquet,/home/jovyan/notebooks/03_data_final/feature_l...,/home/jovyan/notebooks/03_data_final/feature_l...,1.133


Total parquet files: 18


In [6]:
loaded_layers = {}

for _, row in parquet_df.iterrows():
    path = Path(row["full_path"])
    key = path.stem.replace(" ", "_").replace("-", "_")

    try:
        df = pd.read_parquet(path)
        loaded_layers[key] = df
        print(f"Loaded: {key:40s} | shape={df.shape}")
    except Exception as e:
        print(f"FAILED: {path.name} -> {e}")

print(f"\nLoaded {len(loaded_layers)} parquet datasets.")

Loaded: alaska_base_map                          | shape=(9, 169)
Loaded: clean_2d_seismic                         | shape=(26, 7)
Loaded: clean_2d_seismic_features                | shape=(8, 13)
Loaded: clean_3d_seismic                         | shape=(36, 8)
Loaded: clean_3d_seismic_features                | shape=(24, 15)
Loaded: clean_well_locations                     | shape=(10250, 4)
Loaded: clean_well_locations_features            | shape=(33078, 23)
Loaded: clean_well_locations_features_dedup      | shape=(9861, 24)
Loaded: north_slope_assessment_units             | shape=(6, 6)
Loaded: north_slope_assessment_units_features    | shape=(6, 10)
Loaded: north_slope_extent                       | shape=(1, 2)
Loaded: north_slope_extent_features              | shape=(1, 2)
Loaded: north_slope_master_2d_layers             | shape=(538877, 26)
Loaded: north_slope_master_3d_surfaces           | shape=(723840, 8)
Loaded: north_slope_structural_surfaces          | shape=(723840, 9)
Loa

In [7]:
for name, df in loaded_layers.items():
    print("\n" + "=" * 80)
    print(name)
    print(df.head(3))
    print(df.columns.tolist())


alaska_base_map
        featurecla  scalerank  LABELRANK                SOVEREIGNT SOV_A3  \
4  Admin-0 country          1          2  United States of America    US1   
4  Admin-0 country          1          2  United States of America    US1   
4  Admin-0 country          1          2  United States of America    US1   

   ADM0_DIF  LEVEL     TYPE TLC                     ADMIN  ... FCLASS_TR  \
4         1      2  Country   1  United States of America  ...       NaN   
4         1      2  Country   1  United States of America  ...       NaN   
4         1      2  Country   1  United States of America  ...       NaN   

   FCLASS_ID FCLASS_PL FCLASS_GR  FCLASS_IT FCLASS_NL FCLASS_SE  FCLASS_BD  \
4        NaN       NaN       NaN        NaN       NaN       NaN        NaN   
4        NaN       NaN       NaN        NaN       NaN       NaN        NaN   
4        NaN       NaN       NaN        NaN       NaN       NaN        NaN   

  FCLASS_UA                                           ge

In [8]:
from pathlib import Path

ROOT = Path("/home/jovyan/notebooks")
RAW_DIR = ROOT / "raw_data"
DEPTH_GRID_DIR = RAW_DIR / "north_slope_depth_grids"

print("DEPTH_GRID_DIR =", DEPTH_GRID_DIR)
print("Exists?        =", DEPTH_GRID_DIR.exists())
print("Is dir?        =", DEPTH_GRID_DIR.is_dir())

xyz_files = sorted(DEPTH_GRID_DIR.glob("*.XYZ")) + sorted(DEPTH_GRID_DIR.glob("*.xyz"))

print("\nXYZ files found:")
for p in xyz_files:
    print(" -", p.name)

print(f"\nTotal XYZ files: {len(xyz_files)}")

DEPTH_GRID_DIR = /home/jovyan/notebooks/raw_data/north_slope_depth_grids
Exists?        = True
Is dir?        = True

XYZ files found:
 - NSLCU-Shublik.XYZ
 - NSLCU.XYZ
 - NSbasement.XYZ
 - NSshublik-basement.XYZ
 - NSshublik.XYZ
 - NStopo-LCU.XYZ
 - NStopo-basement.XYZ
 - NStopo.XYZ

Total XYZ files: 8


In [9]:
import pandas as pd

def read_xyz_file(path):
    rows = []

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()

            if (
                not line
                or line.startswith("/")
                or ("X" in line and "Y" in line and "Z" in line)
                or "----" in line
            ):
                continue

            parts = line.split()
            if len(parts) < 5:
                continue

            try:
                x = float(parts[0])
                y = float(parts[1])
                z = float(parts[2])
                lon = float(parts[3])
                lat = float(parts[4])
                rows.append((x, y, z, lon, lat))
            except ValueError:
                continue

    if not rows:
        raise ValueError(f"No valid rows found in {path.name}")

    return pd.DataFrame(rows, columns=["x", "y", "z", "lon", "lat"])

In [10]:
test_xyz = DEPTH_GRID_DIR / "NSshublik.XYZ"
test_df = read_xyz_file(test_xyz)

print(test_df.head())
print(test_df.describe())
print("Rows:", len(test_df))

          x          y       z        lon       lat
0 -630000.0  2000000.0  -611.0 -168.67649  67.32879
1 -627000.0  2000000.0  -791.0 -168.60884  67.33470
2 -624000.0  2000000.0 -1092.0 -168.54116  67.34059
3 -621000.0  2000000.0 -1391.0 -168.47344  67.34646
4 -618000.0  2000000.0 -1668.0 -168.40570  67.35229
                   x             y             z           lon           lat
count   90480.000000  9.048000e+04  90480.000000  90480.000000  90480.000000
mean    21000.000000  2.310500e+06  -5446.249050   -153.450434     70.530463
std    376722.137017  1.801322e+05   4674.469126      9.972431      1.646659
min   -630000.000000  2.000000e+06 -16691.000000   -172.657310     67.243160
25%   -306000.000000  2.155250e+06 -10251.000000   -162.088437     69.116290
50%     21000.000000  2.310500e+06  -4526.000000   -153.440745     70.521000
75%    348000.000000  2.465750e+06  -1417.000000   -144.811743     71.936842
max    672000.000000  2.621000e+06   2084.000000   -134.169420     73.65

In [11]:
import numpy as np
import pandas as pd

def estimate_spacing(values):
    vals = np.sort(np.unique(values))
    diffs = np.diff(vals)
    diffs = diffs[diffs > 0]
    if len(diffs) == 0:
        return None
    return float(pd.Series(diffs).mode().iloc[0])

dx = estimate_spacing(test_df["x"].values)
dy = estimate_spacing(test_df["y"].values)

print("Estimated dx:", dx)
print("Estimated dy:", dy)
print("Unique X:", test_df["x"].nunique())
print("Unique Y:", test_df["y"].nunique())

Estimated dx: 3000.0
Estimated dy: 3000.0
Unique X: 435
Unique Y: 208


In [12]:
import rasterio
from rasterio.transform import from_origin

def xyz_to_raster_xy(df, out_tif, crs=None, nodata=-9999.0):
    x_vals = np.sort(df["x"].unique())
    y_vals = np.sort(df["y"].unique())

    dx = estimate_spacing(x_vals)
    dy = estimate_spacing(y_vals)

    if dx is None or dy is None:
        raise ValueError("Could not determine x/y spacing.")

    print(f"Grid size will be {len(y_vals)} rows x {len(x_vals)} cols")

    grid = df.pivot_table(index="y", columns="x", values="z", aggfunc="mean")
    grid = grid.reindex(index=y_vals, columns=x_vals)

    arr = grid.to_numpy(dtype="float32")
    arr = np.where(np.isnan(arr), nodata, arr)

    # north-up raster
    arr = np.flipud(arr)

    west = x_vals.min() - dx / 2
    north = y_vals.max() + dy / 2
    transform = from_origin(west, north, dx, dy)

    with rasterio.open(
        out_tif,
        "w",
        driver="GTiff",
        height=arr.shape[0],
        width=arr.shape[1],
        count=1,
        dtype="float32",
        crs=crs,
        transform=transform,
        nodata=nodata,
    ) as dst:
        dst.write(arr, 1)

    return out_tif

In [13]:
RASTER_OUT_DIR = DEPTH_GRID_DIR / "rasters_from_xyz"
RASTER_OUT_DIR.mkdir(parents=True, exist_ok=True)

out_tif = RASTER_OUT_DIR / "NSshublik_xy.tif"
xyz_to_raster_xy(test_df, out_tif, crs=None)

print("Wrote:", out_tif)

Grid size will be 208 rows x 435 cols
Wrote: /home/jovyan/notebooks/raw_data/north_slope_depth_grids/rasters_from_xyz/NSshublik_xy.tif


In [ ]:
xyz_files = sorted(DEPTH_GRID_DIR.glob("*.XYZ")) + sorted(DEPTH_GRID_DIR.glob("*.xyz"))

raster_outputs = []

for xyz_path in xyz_files:
    try:
        df = read_xyz_file(xyz_path)
        out_tif = RASTER_OUT_DIR / f"{xyz_path.stem}.tif"
        xyz_to_raster_lonlat(df, out_tif)
        raster_outputs.append(out_tif)
        print(f"OK   -> {xyz_path.name} -> {out_tif.name}")
    except Exception as e:
        print(f"FAIL -> {xyz_path.name}: {e}")

print(f"\nCreated {len(raster_outputs)} rasters.")

/tmp/ipykernel_224/1769764110.py:16: PerformanceWarning: The following operation may generate 4079985728 cells in the resulting pandas object.
  grid = df.pivot_table(index="lat", columns="lon", values="z", aggfunc="mean")


In [14]:
xyz_files = sorted(DEPTH_GRID_DIR.glob("*.XYZ")) + sorted(DEPTH_GRID_DIR.glob("*.xyz"))

raster_outputs = []

for xyz_path in xyz_files:
    try:
        df = read_xyz_file(xyz_path)
        out_tif = RASTER_OUT_DIR / f"{xyz_path.stem}_xy.tif"
        xyz_to_raster_xy(df, out_tif, crs=None)
        raster_outputs.append(out_tif)
        print(f"OK   -> {xyz_path.name} -> {out_tif.name}")
    except Exception as e:
        print(f"FAIL -> {xyz_path.name}: {e}")

print(f"\nCreated {len(raster_outputs)} rasters.")

Grid size will be 208 rows x 435 cols
OK   -> NSLCU-Shublik.XYZ -> NSLCU-Shublik_xy.tif
Grid size will be 208 rows x 435 cols
OK   -> NSLCU.XYZ -> NSLCU_xy.tif
Grid size will be 208 rows x 435 cols
OK   -> NSbasement.XYZ -> NSbasement_xy.tif
Grid size will be 208 rows x 435 cols
OK   -> NSshublik-basement.XYZ -> NSshublik-basement_xy.tif
Grid size will be 208 rows x 435 cols
OK   -> NSshublik.XYZ -> NSshublik_xy.tif
Grid size will be 208 rows x 435 cols
OK   -> NStopo-LCU.XYZ -> NStopo-LCU_xy.tif
Grid size will be 208 rows x 435 cols
OK   -> NStopo-basement.XYZ -> NStopo-basement_xy.tif
Grid size will be 208 rows x 435 cols
OK   -> NStopo.XYZ -> NStopo_xy.tif

Created 8 rasters.


In [17]:
from pathlib import Path
import subprocess
import sys

BASE_DIR = Path("/home/jovyan/notebooks")
export_dir = BASE_DIR / "notebook_exports_pdf"
export_dir.mkdir(exist_ok=True)

notebooks = [
    BASE_DIR / "01_base_map_reorganized.ipynb",
    BASE_DIR / "01_parquet_prep_cleaned.ipynb",
    BASE_DIR / "02_geologic_framework.ipynb",
    BASE_DIR / "Depth_grid_processing.ipynb",
    BASE_DIR / "Master.ipynb",
    BASE_DIR / "01_pipeline" / "GISviz.ipynb",
    BASE_DIR / "02_visualization" / "North Slope Data Layer Map.ipynb",
]

for nb_path in notebooks:
    if not nb_path.exists():
        print(f"SKIP: {nb_path} not found")
        continue

    print(f"Exporting {nb_path.name} -> PDF")

    cmd = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "pdf",
        str(nb_path),
        "--output-dir",
        str(export_dir),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"DONE: {nb_path.name}")
    else:
        print(f"FAILED: {nb_path.name}")
        print(result.stderr)

print(f"\nFinished. PDF files are in:\n{export_dir}")

Exporting 01_base_map_reorganized.ipynb -> PDF
DONE: 01_base_map_reorganized.ipynb
Exporting 01_parquet_prep_cleaned.ipynb -> PDF
DONE: 01_parquet_prep_cleaned.ipynb
Exporting 02_geologic_framework.ipynb -> PDF
DONE: 02_geologic_framework.ipynb
Exporting Depth_grid_processing.ipynb -> PDF
DONE: Depth_grid_processing.ipynb
Exporting Master.ipynb -> PDF
DONE: Master.ipynb
Exporting GISviz.ipynb -> PDF
DONE: GISviz.ipynb
Exporting North Slope Data Layer Map.ipynb -> PDF
DONE: North Slope Data Layer Map.ipynb

Finished. PDF files are in:
/home/jovyan/notebooks/notebook_exports_pdf


In [18]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/home/jovyan/notebooks")
MASTER_DIR = BASE_DIR / "03_data_final" / "master_layers"
OUT_DIR = BASE_DIR / "03_data_final" / "gis_ready_surfaces"
OUT_DIR.mkdir(parents=True, exist_ok=True)

master_3d = pd.read_parquet(MASTER_DIR / "north_slope_master_3d_surfaces.parquet")

# keep the 3 main depth layers for now
surface_names = ["NStopo", "NSshublik", "NSbasement"]

for sname in surface_names:
    df = master_3d[master_3d["surface_name"] == sname].copy()

    # keep clean GIS-ready columns
    out = df[[
        "surface_name",
        "source_file",
        "x_3338",
        "y_3338",
        "lon",
        "lat",
        "z_native",
        "depth_m"
    ]].copy()

    # optional: standardize a GIS z field
    out["z_m"] = out["depth_m"]

    out_path = OUT_DIR / f"{sname}_gis_ready.parquet"
    out.to_parquet(out_path, index=False)

    print(f"Saved: {out_path} | rows={len(out)}")
    print(out.head(3))

Saved: /home/jovyan/notebooks/03_data_final/gis_ready_surfaces/NStopo_gis_ready.parquet | rows=90480
  surface_name source_file    x_3338     y_3338         lon        lat  \
0       NStopo  NStopo.XYZ -630000.0  2621000.0 -172.658577  72.824670   
1       NStopo  NStopo.XYZ -627000.0  2621000.0 -172.574298  72.832387   
2       NStopo  NStopo.XYZ -624000.0  2621000.0 -172.489958  72.840071   

   z_native  depth_m   z_m  
0     -67.0     67.0  67.0  
1     -75.0     75.0  75.0  
2     -82.0     82.0  82.0  
Saved: /home/jovyan/notebooks/03_data_final/gis_ready_surfaces/NSshublik_gis_ready.parquet | rows=90480
       surface_name    source_file    x_3338     y_3338         lon  \
180960    NSshublik  NSshublik.XYZ -630000.0  2621000.0 -172.658577   
180961    NSshublik  NSshublik.XYZ -627000.0  2621000.0 -172.574298   
180962    NSshublik  NSshublik.XYZ -624000.0  2621000.0 -172.489958   

              lat  z_native  depth_m      z_m  
180960  72.824670  -16482.0  16482.0  16482.0  
1

In [19]:
import pandas as pd

df_topo = pd.read_parquet("/home/jovyan/notebooks/03_data_final/gis_ready_surfaces/NStopo_gis_ready.parquet")
df_shub = pd.read_parquet("/home/jovyan/notebooks/03_data_final/gis_ready_surfaces/NSshublik_gis_ready.parquet")
df_base = pd.read_parquet("/home/jovyan/notebooks/03_data_final/gis_ready_surfaces/NSbasement_gis_ready.parquet")

print(df_topo.head())
print(df_shub.head())
print(df_base.head())

  surface_name source_file    x_3338     y_3338         lon        lat  \
0       NStopo  NStopo.XYZ -630000.0  2621000.0 -172.658577  72.824670   
1       NStopo  NStopo.XYZ -627000.0  2621000.0 -172.574298  72.832387   
2       NStopo  NStopo.XYZ -624000.0  2621000.0 -172.489958  72.840071   
3       NStopo  NStopo.XYZ -621000.0  2621000.0 -172.405556  72.847722   
4       NStopo  NStopo.XYZ -618000.0  2621000.0 -172.321093  72.855338   

   z_native  depth_m   z_m  
0     -67.0     67.0  67.0  
1     -75.0     75.0  75.0  
2     -82.0     82.0  82.0  
3     -81.0     81.0  81.0  
4     -74.0     74.0  74.0  
  surface_name    source_file    x_3338     y_3338         lon        lat  \
0    NSshublik  NSshublik.XYZ -630000.0  2621000.0 -172.658577  72.824670   
1    NSshublik  NSshublik.XYZ -627000.0  2621000.0 -172.574298  72.832387   
2    NSshublik  NSshublik.XYZ -624000.0  2621000.0 -172.489958  72.840071   
3    NSshublik  NSshublik.XYZ -621000.0  2621000.0 -172.405556  72.847722

In [20]:
import geopandas as gpd

def to_gdf(df):
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )
    return gdf

gdf_topo = to_gdf(df_topo)
gdf_shub = to_gdf(df_shub)
gdf_base = to_gdf(df_base)

In [21]:
gdf_topo.to_file("/home/jovyan/notebooks/03_data_final/gis_ready_surfaces/NStopo_points.geojson", driver="GeoJSON")
gdf_shub.to_file("/home/jovyan/notebooks/03_data_final/gis_ready_surfaces/NSshublik_points.geojson", driver="GeoJSON")
gdf_base.to_file("/home/jovyan/notebooks/03_data_final/gis_ready_surfaces/NSbasement_points.geojson", driver="GeoJSON")

In [24]:
import geopandas as gpd
from jupytergis import GISDocument

# Create GIS document (this connects to the jGIS UI)
doc = GISDocument()

# Load your 3 layers
topo = gpd.read_file("/home/jovyan/notebooks/03_data_final/gis_ready_surfaces/NStopo_points.geojson")
shub = gpd.read_file("/home/jovyan/notebooks/03_data_final/gis_ready_surfaces/NSshublik_points.geojson")
base = gpd.read_file("/home/jovyan/notebooks/03_data_final/gis_ready_surfaces/NSbasement_points.geojson")

# Add layers
doc.add_vector_layer(topo, name="Topography")
doc.add_vector_layer(shub, name="Shublik Surface")
doc.add_vector_layer(base, name="Basement")

doc

AttributeError: 'GISDocument' object has no attribute 'add_file'